# 8. Model Optimisation and Validation

## 8.1 Objective

The primary objective of this section is to perform hyperparameter optimization on candidate classifiers (Logistic Regression, Linear SVM, Random Forest, and Extra Trees) to predict post engagement performance classes (`Low`, `Medium`, `High`) using pre-publication characteristics. We isolate training distributions from testing slices to prevent train/test contamination and evaluate performance separately on Combined, Synthetic, and Real test observations.


In [1]:
import os
import sys
import time
import json
import pickle
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import joblib

# Configure stdout encoding to utf-8
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# Robust Project root setup
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"
BEST_MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOT_DIR = RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)


Project root: D:\newwwwwwww\AiBasedInstagramPrediction
Processed data directory: D:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed
Results directory: D:\newwwwwwww\AiBasedInstagramPrediction\results


## 8.2 Optimisation Methodology

We employ stratified 5-fold cross-validation (`StratifiedKFold`) during the hyperparameter tuning phase. To optimize across multiple features while remaining computationally efficient, we use `RandomizedSearchCV` with `n_iter=3`, utilizing `Weighted F1` as the primary optimization metric for multi-class classification.


## 8.3 Dataset Preparation

We load the preprocessed sparse training/testing matrices, labels, target encoders, and data-source masks, verifying shapes and dimensions.


In [2]:
# Load preprocessed arrays and labels
X_train = scipy.sparse.load_npz(PROCESSED_DIR / "X_train.npz")
X_test = scipy.sparse.load_npz(PROCESSED_DIR / "X_test.npz")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")['target']
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")['target']
real_mask_train = np.load(PROCESSED_DIR / "real_mask_train.npy")
real_mask_test = np.load(PROCESSED_DIR / "real_mask_test.npy")

with open(MODEL_DIR / "target_encoder.pkl", "rb") as f:
    target_mapping = pickle.load(f)

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    tfidf_dict = pickle.load(f)

with open(MODEL_DIR / "preprocessor.pkl", "rb") as f:
    preprocessors = pickle.load(f)

# Reconstruct feature names list
tfidf_cap = tfidf_dict['caption']
tfidf_hash = tfidf_dict['hashtags']
ohe = preprocessors['ohe']

feature_names = []
feature_names.extend([f"caption_{name}" for name in tfidf_cap.get_feature_names_out()])
feature_names.extend([f"hashtag_{name}" for name in tfidf_hash.get_feature_names_out()])
feature_names.extend(['hour_sin', 'hour_cos'])
feature_names.extend(['day_sin', 'day_cos'])
feature_names.extend(['log_follower_count'])
feature_names.extend(['verified_status', 'sponsored', 'is_weekend'])
ohe_categories = list(ohe.get_feature_names_out(['media_type']))
feature_names.extend(ohe_categories)
feature_names.extend(['caption_length', 'word_count', 'hashtag_count'])
feature_names = np.array(feature_names)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(f"Number of reconstructed features: {len(feature_names)}")


X_train shape: (81600, 1517), y_train shape: (81600,)
X_test shape: (20400, 1517), y_test shape: (20400,)
Number of reconstructed features: 1517


## 8.4 Leakage Verification

We run a programmatic check on feature names to ensure no post-publication metrics or direct identifiers are included in the predictor columns, outputting a clear audit table.


In [3]:
leak_vars = ['likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate', 'binary_performance', 'post_id', 'account_id']
exact_leaks = [v for v in leak_vars if v in feature_names]

audit_records = []
for v in leak_vars:
    present_in_X = "Yes" if v in feature_names else "No"
    used_as_pred = "Yes" if present_in_X == "Yes" else "No"
    reason = "Target variable" if "performance" in v else "Unique identifier key" if "id" in v else "Post-publication outcome metric"
    audit_records.append({
        "Column": v,
        "Present": present_in_X,
        "Used as Predictor": used_as_pred,
        "Reason": reason
    })

audit_df = pd.DataFrame(audit_records)
display(audit_df)

if len(exact_leaks) == 0:
    print("Leakage check passed")
else:
    print("Leakage check failed. LEAKS DETECTED:", exact_leaks)
    sys.exit(1)


               Column  ...                           Reason
0               likes  ...  Post-publication outcome metric
1            comments  ...  Post-publication outcome metric
2              shares  ...  Post-publication outcome metric
3               saves  ...  Post-publication outcome metric
4               reach  ...  Post-publication outcome metric
5         impressions  ...  Post-publication outcome metric
6     engagement_rate  ...  Post-publication outcome metric
7  binary_performance  ...                  Target variable
8             post_id  ...            Unique identifier key
9          account_id  ...            Unique identifier key

[10 rows x 4 columns]
Leakage check passed


## 8.5 Cross-Validation Strategy

We define the stratified cross-validation setup used across all hyperparameter searches.


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Stratified 5-fold CV configured with random_state 42")


Stratified 5-fold CV configured with random_state 42


## 8.6 Logistic Regression Optimisation

We optimize Logistic Regression parameters using a randomized search over regularization boundaries.


In [5]:
lr_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs']
}

print("Running Logistic Regression hyperparameter search...")
lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_distributions=lr_grid,
    n_iter=3,
    cv=cv,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
lr_search.fit(X_train, y_train)
lr_time = time.time() - t0

print(f"Logistic Regression optimization complete in {lr_time:.2f}s")
print(f"Best parameters: {lr_search.best_params_}")
print(f"Best weighted CV F1: {lr_search.best_score_:.4f}")


Running Logistic Regression hyperparameter search...
Logistic Regression optimization complete in 43.02s
Best parameters: {'solver': 'lbfgs', 'C': 0.01}
Best weighted CV F1: 0.4458


## 8.7 Linear SVM Optimisation

We optimize LinearSVC margins and parameters.


In [6]:
svm_grid = {
    'C': [0.01, 0.1, 1.0]
}

print("Running Linear SVM hyperparameter search...")
svm_search = RandomizedSearchCV(
    LinearSVC(random_state=42, max_iter=2000),
    param_distributions=svm_grid,
    n_iter=3,
    cv=cv,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
svm_search.fit(X_train, y_train)
svm_time = time.time() - t0

print(f"Linear SVM optimization complete in {svm_time:.2f}s")
print(f"Best parameters: {svm_search.best_params_}")
print(f"Best weighted CV F1: {svm_search.best_score_:.4f}")


Running Linear SVM hyperparameter search...
Linear SVM optimization complete in 79.67s
Best parameters: {'C': 0.1}
Best weighted CV F1: 0.4435


## 8.8 Random Forest Optimisation

We run a randomized search on forest depths and split requirements.


In [7]:
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

print("Running Random Forest hyperparameter search...")
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_grid,
    n_iter=3,
    cv=cv,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf_search.fit(X_train, y_train)
rf_time = time.time() - t0

print(f"Random Forest optimization complete in {rf_time:.2f}s")
print(f"Best parameters: {rf_search.best_params_}")
print(f"Best weighted CV F1: {rf_search.best_score_:.4f}")


Running Random Forest hyperparameter search...
Random Forest optimization complete in 75.02s
Best parameters: {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 20}
Best weighted CV F1: 0.4299


## 8.9 Extra Trees Optimisation

We optimize ExtraTreesClassifier settings.


In [8]:
et_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20]
}

print("Running Extra Trees hyperparameter search...")
et_search = RandomizedSearchCV(
    ExtraTreesClassifier(random_state=42, n_jobs=-1),
    param_distributions=et_grid,
    n_iter=3,
    cv=cv,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
et_search.fit(X_train, y_train)
et_time = time.time() - t0

print(f"Extra Trees optimization complete in {et_time:.2f}s")
print(f"Best parameters: {et_search.best_params_}")
print(f"Best weighted CV F1: {et_search.best_score_:.4f}")


Running Extra Trees hyperparameter search...
Extra Trees optimization complete in 78.62s
Best parameters: {'n_estimators': 200, 'max_depth': 20}
Best weighted CV F1: 0.4161


## 8.10 Model Comparison

We compile and evaluate the optimized estimators on the held-out combined test dataset, reporting accuracy, precision, recall, and weighted F1 metrics.


In [9]:
opt_results = []

def evaluate_opt_model(search_obj, name):
    best_est = search_obj.best_estimator_
    
    t0 = time.time()
    y_pred = best_est.predict(X_test)
    pred_time = time.time() - t0
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    
    opt_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "Weighted_F1": round(f1, 4),
        "Best_Parameters": str(search_obj.best_params_),
        "y_pred": y_pred
    })
    print(f"{name} -> Accuracy: {acc:.4f} | Weighted F1: {f1:.4f}")

evaluate_opt_model(lr_search, "Optimized Logistic Regression")
evaluate_opt_model(svm_search, "Optimized Linear SVM")
evaluate_opt_model(rf_search, "Optimized Random Forest")
evaluate_opt_model(et_search, "Optimized Extra Trees")

comparison_df = pd.DataFrame(opt_results)
display(comparison_df.drop(columns=['y_pred']).sort_values("Weighted_F1", ascending=False))

comparison_df.drop(columns=['y_pred']).to_csv(RESULTS_DIR / "model_optimization_results.csv", index=False)
print("Saved model_optimization_results.csv")


Optimized Logistic Regression -> Accuracy: 0.4495 | Weighted F1: 0.4443
Optimized Linear SVM -> Accuracy: 0.4500 | Weighted F1: 0.4420
Optimized Random Forest -> Accuracy: 0.4439 | Weighted F1: 0.4284
Optimized Extra Trees -> Accuracy: 0.4415 | Weighted F1: 0.4177
                           Model  ...                                    Best_Parameters
0  Optimized Logistic Regression  ...                     {'solver': 'lbfgs', 'C': 0.01}
1           Optimized Linear SVM  ...                                         {'C': 0.1}
2        Optimized Random Forest  ...  {'n_estimators': 200, 'min_samples_split': 2, ...
3          Optimized Extra Trees  ...             {'n_estimators': 200, 'max_depth': 20}

[4 rows x 6 columns]
Saved model_optimization_results.csv


## 8.11 Cross-Validation Results

We extract and report cross-validation standard deviation results for the optimized search runs.


In [10]:
cv_summary = [
    {"Model": "Logistic Regression", "Mean_CV_F1": lr_search.best_score_, "CV_F1_Std": lr_search.cv_results_['std_test_score'][lr_search.best_index_]},
    {"Model": "Linear SVM", "Mean_CV_F1": svm_search.best_score_, "CV_F1_Std": svm_search.cv_results_['std_test_score'][svm_search.best_index_]},
    {"Model": "Random Forest", "Mean_CV_F1": rf_search.best_score_, "CV_F1_Std": rf_search.cv_results_['std_test_score'][rf_search.best_index_]},
    {"Model": "Extra Trees", "Mean_CV_F1": et_search.best_score_, "CV_F1_Std": et_search.cv_results_['std_test_score'][et_search.best_index_]}
]

cv_summary_df = pd.DataFrame(cv_summary)
display(cv_summary_df)

cv_summary_df.to_csv(RESULTS_DIR / "model_optimization_cv_results.csv", index=False)
print("Saved model_optimization_cv_results.csv")


                 Model  Mean_CV_F1  CV_F1_Std
0  Logistic Regression    0.445799   0.003774
1           Linear SVM    0.443456   0.004182
2        Random Forest    0.429924   0.002383
3          Extra Trees    0.416123   0.001453
Saved model_optimization_cv_results.csv


## 8.12 Best Model Selection

We select the best optimized classifier based on its generalization metrics, cross-validation stability, and performance on the evaluation splits.


In [11]:
# Select best based on highest Weighted F1
best_idx = comparison_df['Weighted_F1'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_model_pred = comparison_df.loc[best_idx, 'y_pred']

print(f"Selected Best Model: {best_model_name}")

if best_model_name == "Optimized Logistic Regression":
    best_estimator = lr_search.best_estimator_
    best_params = lr_search.best_params_
elif best_model_name == "Optimized Linear SVM":
    best_estimator = svm_search.best_estimator_
    best_params = svm_search.best_params_
elif best_model_name == "Optimized Random Forest":
    best_estimator = rf_search.best_estimator_
    best_params = rf_search.best_params_
else:
    best_estimator = et_search.best_estimator_
    best_params = et_search.best_params_

with open(RESULTS_DIR / "best_model_parameters.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("Saved best_model_parameters.json")


Selected Best Model: Optimized Logistic Regression
Saved best_model_parameters.json


## 8.13 Confusion Matrix

We save a professional confusion display for the selected model on the test dataset.


In [12]:
class_labels = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test, best_model_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title(f"{best_model_name} Confusion Matrix")
plt.tight_layout()
plt.savefig(PLOT_DIR / "model_optimization_confusion_matrix.png")
plt.close()
print("Saved model_optimization_confusion_matrix.png")


Saved model_optimization_confusion_matrix.png


## 8.14 Error Analysis

We inspect the class-to-class confusions for our best classifier, comparing error rates between tiers.


In [13]:
# Calculate percentage of misclassifications
misclass_mask = y_test != best_model_pred
total_errors = np.sum(misclass_mask)
print(f"Total misclassifications: {total_errors} out of {len(y_test)} ({total_errors/len(y_test)*100:.2f}%)")

# Specific confusions
low_as_med = np.sum((y_test == 0) & (best_model_pred == 1))
low_as_high = np.sum((y_test == 0) & (best_model_pred == 2))
med_as_low = np.sum((y_test == 1) & (best_model_pred == 0))
med_as_high = np.sum((y_test == 1) & (best_model_pred == 2))
high_as_low = np.sum((y_test == 2) & (best_model_pred == 0))
high_as_med = np.sum((y_test == 2) & (best_model_pred == 1))

print(f"  Low performance posts predicted as Medium: {low_as_med}")
print(f"  Low performance posts predicted as High: {low_as_high}")
print(f"  Medium performance posts predicted as Low: {med_as_low}")
print(f"  Medium performance posts predicted as High: {med_as_high}")
print(f"  High performance posts predicted as Low: {high_as_low}")
print(f"  High performance posts predicted as Medium: {high_as_med}")


Total misclassifications: 11230 out of 20400 (55.05%)
  Low performance posts predicted as Medium: 1780
  Low performance posts predicted as High: 1803
  Medium performance posts predicted as Low: 2334
  Medium performance posts predicted as High: 2263
  High performance posts predicted as Low: 1714
  High performance posts predicted as Medium: 1336


## 8.15 Real-vs-Synthetic Generalisation

We evaluate our best optimized classifier separately on Real and Synthetic test data slices to measure domain shift boundaries.


In [14]:
# Real observations evaluation
y_test_real = y_test[real_mask_test]
pred_real = best_model_pred[real_mask_test]

acc_r = accuracy_score(y_test_real, pred_real)
prec_r, rec_r, f1_r, _ = precision_recall_fscore_support(y_test_real, pred_real, average='weighted', zero_division=0)

# Synthetic observations evaluation
y_test_synth = y_test[~real_mask_test]
pred_synth = best_model_pred[~real_mask_test]

acc_s = accuracy_score(y_test_synth, pred_synth)
prec_s, rec_s, f1_s, _ = precision_recall_fscore_support(y_test_synth, pred_synth, average='weighted', zero_division=0)

generalisation_records = [
    {"Dataset": "Real test observations only", "Accuracy": round(acc_r, 4), "Precision": round(prec_r, 4), "Recall": round(rec_r, 4), "Weighted_F1": round(f1_r, 4)},
    {"Dataset": "Synthetic test observations only", "Accuracy": round(acc_s, 4), "Precision": round(prec_s, 4), "Recall": round(rec_s, 4), "Weighted_F1": round(f1_s, 4)},
    {"Dataset": "Combined test dataset", "Accuracy": comparison_df.loc[best_idx, 'Accuracy'], "Precision": comparison_df.loc[best_idx, 'Precision'], "Recall": comparison_df.loc[best_idx, 'Recall'], "Weighted_F1": comparison_df.loc[best_idx, 'Weighted_F1']}
]

generalisation_df = pd.DataFrame(generalisation_records)
display(generalisation_df)

generalisation_df.to_csv(RESULTS_DIR / "model_optimization_summary.csv", index=False)
print("Saved model_optimization_summary.csv")


                            Dataset  Accuracy  Precision  Recall  Weighted_F1
0       Real test observations only    0.4227     0.4499  0.4227       0.4232
1  Synthetic test observations only    0.4500     0.4452  0.4500       0.4444
2             Combined test dataset    0.4495     0.4447  0.4495       0.4443
Saved model_optimization_summary.csv


## 8.16 Feature/Model Interpretation

We map model parameters and Gini feature importances back to their pre-publication text-derived and metadata names, highlighting the top 20 predictors.


In [15]:
if hasattr(best_estimator, 'coef_'):
    # Linear model coefficients
    coef_mag = np.mean(np.abs(best_estimator.coef_), axis=0)
    interpret_df = pd.DataFrame({
        "Feature": feature_names,
        "Magnitude": coef_mag
    }).sort_values("Magnitude", ascending=False)
else:
    # Tree model importances
    interpret_df = pd.DataFrame({
        "Feature": feature_names,
        "Magnitude": best_estimator.feature_importances_
    }).sort_values("Magnitude", ascending=False)

print("Top 20 most important features:")
display(interpret_df.head(20))


Top 20 most important features:
                    Feature  Magnitude
1512        media_type_Reel   0.321130
1510       media_type_Photo   0.287584
846            caption_this   0.164101
959            caption_with   0.153262
963       caption_with your   0.143822
285             caption_for   0.133480
992            caption_your   0.125125
983             caption_you   0.116957
725           caption_share   0.102633
703       caption_save this   0.098069
381              caption_if   0.097853
134   caption_comment below   0.096827
69       caption_below with   0.096558
133         caption_comment   0.095483
701            caption_save   0.094835
382          caption_if you   0.094438
1505        verified_status   0.091324
135        caption_comments   0.090785
1504     log_follower_count   0.090194
1136       hashtag_foodporn   0.088330


## 8.17 Academic Summary

The model optimization and validation stage is completed. Below is the programmatic summary of the results:


In [16]:
summary_info = {
    "Models_Evaluated": ["Logistic Regression", "Linear SVM", "Random Forest", "Extra Trees"],
    "Optimisation_Method": "RandomizedSearchCV",
    "Cross_Validation_Configuration": "Stratified 5-Fold",
    "Best_Model": best_model_name,
    "Best_Weighted_F1": float(comparison_df.loc[best_idx, 'Weighted_F1']),
    "Best_Accuracy": float(comparison_df.loc[best_idx, 'Accuracy']),
    "Real_Data_Performance_F1": round(f1_r, 4),
    "Synthetic_Data_Performance_F1": round(f1_s, 4),
    "Domain_Shift_Observation": "Real-data F1 is slightly lower than synthetic-data F1, indicating domain shift",
    "Leakage_Verification_Result": "Leakage check passed" if len(exact_leaks) == 0 else "Leakage check failed",
    "Important_Limitations": "Absence of visual modalities for real scraped posts limits multimodal alignment",
    "Next_Recommended_Research_Stage": "Model deployment or secondary visual multimodal trials"
}

print(json.dumps(summary_info, indent=4))

print("\nMODEL OPTIMISATION COMPLETED")
print("NEXT STEP: READY FOR FINAL MODEL SELECTION AND IMAGE-ENHANCED EXPERIMENT.")


{
    "Models_Evaluated": [
        "Logistic Regression",
        "Linear SVM",
        "Random Forest",
        "Extra Trees"
    ],
    "Optimisation_Method": "RandomizedSearchCV",
    "Cross_Validation_Configuration": "Stratified 5-Fold",
    "Best_Model": "Optimized Logistic Regression",
    "Best_Weighted_F1": 0.4443,
    "Best_Accuracy": 0.4495,
    "Real_Data_Performance_F1": 0.4232,
    "Synthetic_Data_Performance_F1": 0.4444,
    "Domain_Shift_Observation": "Real-data F1 is slightly lower than synthetic-data F1, indicating domain shift",
    "Leakage_Verification_Result": "Leakage check passed",
    "Important_Limitations": "Absence of visual modalities for real scraped posts limits multimodal alignment",
    "Next_Recommended_Research_Stage": "Model deployment or secondary visual multimodal trials"
}

MODEL OPTIMISATION COMPLETED
NEXT STEP: READY FOR FINAL MODEL SELECTION AND IMAGE-ENHANCED EXPERIMENT.
